# RAG Data Schema (CSV Target)
저장할 CSV는 다음 열(Column)을 포함해야 합니다:
1. `post_id`: 게시글 고유 번호
2. `url`: 게시글 원본 URL
3. `category`: 게시글 카테고리/머리말 (존재할 경우)
4. `title`: 게시글 제목
5. `author`: 작성자
6. `created_at`: 작성 일시 (YYYY-MM-DD HH:MM:SS 형식 정형화)
7. `views`: 조회수 (숫자 타입)
8. `likes`: 추천수 (숫자 타입)
9. `content`: 본문 텍스트 (RAG 정제 규칙 적용)
10. `comments`: 댓글 데이터 (JSON 문자열 구조: `[{"author": "...", "content": "...", "created_at": "..."}]`)
11. `crawled_at`: 크롤링 일시

In [1]:
import asyncio
import csv
import json
import logging
import os
import random
import re
import sys
import threading
from datetime import datetime
from typing import Dict, List, Optional, Set
from bs4 import BeautifulSoup
from playwright.async_api import BrowserContext, Page, async_playwright

# ==========================================
# 1. 로깅 및 환경 설정
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("MapleInvenScraper")

BOARD_ID = "2300"
BASE_URL = f"https://www.inven.co.kr/board/maple/{BOARD_ID}"
CSV_FILENAME = "maple_inven_tips_rag.csv"
CHECKPOINT_INTERVAL = 5
START_PAGE = 1
END_PAGE = 55

FIELDNAMES = [
    "post_id", "url", "category", "title", "author",
    "created_at", "views", "likes", "content", "comments", "crawled_at"
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0"
]

# ==========================================
# 2. 헬퍼 함수 및 정제 로직
# ==========================================
def random_delay(min_sec: float = 1.5, max_sec: float = 3.0):
    return asyncio.sleep(random.uniform(min_sec, max_sec))

def load_existing_post_ids(csv_path: str) -> Set[str]:
    collected_ids = set()
    if os.path.exists(csv_path):
        try:
            with open(csv_path, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("post_id"):
                        collected_ids.add(str(row["post_id"]).strip())
            logger.info(f"기존 저장된 게시글 {len(collected_ids)}개를 발견하여 스킵 대상에 추가했습니다.")
        except Exception as e:
            logger.warning(f"기존 CSV 로드 중 오류 발생: {e}")
    return collected_ids

def parse_number(text: Optional[str]) -> int:
    if not text:
        return 0
    clean_str = re.sub(r"[^\d]", "", text)
    return int(clean_str) if clean_str.isdigit() else 0

def format_datetime(date_str: Optional[str]) -> str:
    if not date_str:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    date_str = date_str.strip()
    patterns = [
        ("%y-%m-%d %H:%M:%S", r"^\d{2}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        ("%y-%m-%d %H:%M", r"^\d{2}-\d{2}-\d{2} \d{2}:\d{2}$"),
        ("%Y-%m-%d %H:%M:%S", r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        ("%Y-%m-%d %H:%M", r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$"),
        ("%Y.%m.%d %H:%M", r"^\d{4}\.\d{2}\.\d{2} \d{2}:\d{2}$"),
    ]
    for fmt, regex in patterns:
        if re.match(regex, date_str):
            try:
                dt = datetime.strptime(date_str, fmt)
                if dt.year < 2000:
                    dt = dt.replace(year=dt.year + 2000)
                return dt.strftime("%Y-%m-%d %H:%M:%S")
            except ValueError:
                pass
    if re.match(r"^\d{2}:\d{2}$", date_str):
        today_prefix = datetime.now().strftime("%Y-%m-%d")
        return f"{today_prefix} {date_str}:00"
    return date_str

def clean_rag_content(html_content: str) -> str:
    soup = BeautifulSoup(html_content, "html.parser")
    for selector in ["script", "style", "iframe", ".articleAd", ".powerbbs-content-btn", ".articleVote", ".recommend-box", "button"]:
        for tag in soup.select(selector):
            tag.decompose()
    for img in soup.find_all("img"):
        img.replace_with(" [이미지] ")
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    non_empty_lines = [line for line in lines if line]
    cleaned_text = "\n".join(non_empty_lines)
    return re.sub(r"\n{3,}", "\n\n", cleaned_text)

# ==========================================
# 3. 크롤러 클래스
# ==========================================
class MapleInvenScraper:
    def __init__(self, output_file: str = CSV_FILENAME):
        self.output_file = output_file
        self.collected_ids = load_existing_post_ids(self.output_file)
        self.init_csv()

    def init_csv(self):
        if not os.path.exists(self.output_file):
            with open(self.output_file, mode="w", encoding="utf-8-sig", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
                writer.writeheader()
            logger.info(f"신규 CSV 파일 생성: {self.output_file}")

    def save_batch(self, records: List[Dict]):
        if not records:
            return
        with open(self.output_file, mode="a", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
            for rec in records:
                writer.writerow(rec)
        logger.info(f"✅ [체크포인트] {len(records)}개 데이터 저장 완료. (누적: {len(self.collected_ids)}개)")

    async def extract_comments(self, page: Page) -> List[Dict]:
        comments_data = []
        try:
            more_button = page.locator("#cmtDetail .more_cmt_btn, .comment_more_btn")
            if await more_button.is_visible():
                try:
                    await more_button.click(timeout=3000)
                    await page.wait_for_timeout(1000)
                except Exception:
                    pass

            cmt_items = await page.locator(".cmt-list .cmt_item, #cmtDetail .cmtRow, .comment_list li").all()
            for cmt in cmt_items:
                try:
                    author_el = cmt.locator(".writer, .nick, .cmtWriter")
                    author = (await author_el.inner_text()).strip() if await author_el.count() > 0 else "익명"

                    content_el = cmt.locator(".content, .cmtContent, .cmt_text")
                    content = (await content_el.inner_text()).strip() if await content_el.count() > 0 else ""

                    date_el = cmt.locator(".date, .cmtDate, .time")
                    created_at = (await date_el.inner_text()).strip() if await date_el.count() > 0 else ""

                    class_attr = await cmt.get_attribute("class") or ""
                    is_reply = "reply" in class_attr.lower() or "re_cmt" in class_attr.lower() or "depth" in class_attr.lower()

                    if content:
                        comments_data.append({
                            "author": author,
                            "content": content,
                            "created_at": format_datetime(created_at),
                            "is_reply": is_reply
                        })
                except Exception:
                    continue
        except Exception as e:
            logger.debug(f"댓글 추출 중 예외 발생: {e}")
        return comments_data

    async def parse_post_detail(self, context: BrowserContext, post_info: Dict) -> Optional[Dict]:
        page = await context.new_page()
        url = post_info["url"]
        post_id = post_info["post_id"]

        try:
            response = await page.goto(url, wait_until="domcontentloaded", timeout=15000)
            if not response or response.status >= 400:
                await page.close()
                return None

            await random_delay(0.5, 1.0)
            body_text = await page.content()
            if "삭제된 게시글" in body_text or "존재하지 않는" in body_text:
                await page.close()
                return None

            category = post_info.get("category", "")
            if not category:
                cate_el = page.locator(".articleCategory, .board-title .cate")
                if await cate_el.count() > 0:
                    category = (await cate_el.first.inner_text()).strip()

            title = post_info.get("title", "")
            if not title:
                title_el = page.locator(".articleTitle h1, .articleSubject h1, .board-title h1")
                if await title_el.count() > 0:
                    title = (await title_el.first.inner_text()).strip()

            author = post_info.get("author", "")
            if not author:
                author_el = page.locator(".articleWriter, .writer, .nick")
                if await author_el.count() > 0:
                    author = (await author_el.first.inner_text()).strip()

            date_str = ""
            date_el = page.locator(".articleDate, .date, .time")
            if await date_el.count() > 0:
                date_str = (await date_el.first.inner_text()).strip()
            created_at = format_datetime(date_str)

            views = post_info.get("views", 0)
            likes = post_info.get("likes", 0)

            content_el = page.locator("#powerbbsContent, #articleBody, .articleContent")
            raw_html = await content_el.first.inner_html() if await content_el.count() > 0 else await page.content()
            cleaned_content = clean_rag_content(raw_html)

            comments_list = await self.extract_comments(page)
            comments_json = json.dumps(comments_list, ensure_ascii=False)

            await page.close()

            return {
                "post_id": post_id,
                "url": url,
                "category": category,
                "title": title,
                "author": author,
                "created_at": created_at,
                "views": views,
                "likes": likes,
                "content": cleaned_content,
                "comments": comments_json,
                "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
        except Exception as e:
            logger.error(f"게시글 파싱 에러 [{post_id} - {url}]: {e}")
            await page.close()
            return None

    async def fetch_page_posts(self, page: Page, page_num: int) -> List[Dict]:
        """목록 페이지에서 정교한 패턴 분석으로 게시글 링크 추출"""
        page_url = f"{BASE_URL}?p={page_num}"
        logger.info(f"📌 [목록 수집] {page_num} 페이지 접근: {page_url}")
        posts_to_crawl = []

        try:
            await page.goto(page_url, wait_until="domcontentloaded", timeout=20000)
            await random_delay(1.0, 2.0)

            rows = await page.locator("table.board-list tbody tr, .board_list tr, .board-list tbody tr").all()
            for row in rows:
                try:
                    class_name = await row.get_attribute("class") or ""
                    if "notice" in class_name.lower():
                        continue

                    links = await row.locator("a").all()
                    target_link = None
                    post_id = None
                    full_url = ""

                    for link in links:
                        h = await link.get_attribute("href") or ""
                        # /2304/1234567 형태의 게시글 주소만 정확히 추출 (카테고리 링크 /2304?category= 스킵)
                        match = re.search(rf"/{BOARD_ID}/(\d+)", h) or re.search(rf"come_idx={BOARD_ID}.*?&l=(\d+)", h)
                        if match:
                            post_id = match.group(1)
                            target_link = link
                            full_url = h if h.startswith("http") else f"https://www.inven.co.kr{h}"
                            break

                    if not target_link or not post_id:
                        continue

                    if post_id == BOARD_ID or post_id in self.collected_ids:
                        continue

                    title = (await target_link.inner_text()).strip()

                    cate_el = row.locator("td.cate, .category")
                    category = (await cate_el.inner_text()).strip() if await cate_el.count() > 0 else ""

                    writer_el = row.locator("td.user, .writer, .nick")
                    author = (await writer_el.first.inner_text()).strip() if await writer_el.count() > 0 else ""

                    hit_el = row.locator("td.hit, .views")
                    views = parse_number(await hit_el.inner_text()) if await hit_el.count() > 0 else 0

                    recom_el = row.locator("td.recom, .recom")
                    likes = parse_number(await recom_el.inner_text()) if await recom_el.count() > 0 else 0

                    posts_to_crawl.append({
                        "post_id": post_id,
                        "url": full_url,
                        "category": category,
                        "title": title,
                        "author": author,
                        "views": views,
                        "likes": likes
                    })
                except Exception:
                    continue
        except Exception as e:
            logger.error(f"페이지 목록 수집 실패 (Page {page_num}): {e}")

        return posts_to_crawl

    async def run(self, start_page: int = 1, end_page: int = 5):
        async with async_playwright() as p:
            #browser = await p.chromium.launch(headless=True)
            browser = await p.chromium.launch(headless=True, channel="chrome")
            context = await browser.new_context(
                user_agent=random.choice(USER_AGENTS),
                viewport={"width": 1280, "height": 800},
                java_script_enabled=True
            )
            await context.route("**/*.{png,jpg,jpeg,gif,svg,webp,woff,woff2,ttf}", lambda route: route.abort())

            list_page = await context.new_page()
            buffer = []

            for p_num in range(start_page, end_page + 1):
                posts = await self.fetch_page_posts(list_page, p_num)
                logger.info(f"Page {p_num}: {len(posts)}개의 신규 게시글 수집 대상 발견")

                for p_info in posts:
                    post_id = p_info["post_id"]
                    logger.info(f"🔎 [게시글 파싱 중] ID: {post_id} | {p_info['title'][:20]}...")

                    record = await self.parse_post_detail(context, p_info)
                    if record:
                        buffer.append(record)
                        self.collected_ids.add(post_id)

                    if len(buffer) >= CHECKPOINT_INTERVAL:
                        self.save_batch(buffer)
                        buffer.clear()

                    await random_delay(1.0, 2.0)

            if buffer:
                self.save_batch(buffer)
                buffer.clear()

            await browser.close()
            logger.info("🎉 크롤링 작업이 성공적으로 완료되었습니다.")

# ==========================================
# 4. 주피터 실행 함수
# ==========================================
def run_scraper_in_thread(start_page: int = 1, end_page: int = 2):
    def worker():
        if sys.platform == 'win32':
            loop = asyncio.WindowsProactorEventLoopPolicy().new_event_loop()
        else:
            loop = asyncio.new_event_loop()
        
        asyncio.set_event_loop(loop)
        scraper = MapleInvenScraper(output_file=CSV_FILENAME)
        try:
            loop.run_until_complete(scraper.run(start_page=start_page, end_page=end_page))
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()

# 스크래퍼 실행
run_scraper_in_thread(start_page=START_PAGE, end_page=END_PAGE)

2026-08-14 15:34:51,057 [INFO] 기존 저장된 게시글 270개를 발견하여 스킵 대상에 추가했습니다.
2026-08-14 15:34:51,612 [INFO] 📌 [목록 수집] 1 페이지 접근: https://www.inven.co.kr/board/maple/2300?p=1
2026-08-14 15:34:54,866 [INFO] Page 1: 2개의 신규 게시글 수집 대상 발견
2026-08-14 15:34:54,866 [INFO] 🔎 [게시글 파싱 중] ID: 361726 | [아이템] 이번 마라벨 풀셋 가격?...
2026-08-14 15:34:58,138 [INFO] 🔎 [게시글 파싱 중] ID: 361725 | [아이템] 제네무기 추옵이랑 에디 건...
2026-08-14 15:35:01,279 [INFO] 📌 [목록 수집] 2 페이지 접근: https://www.inven.co.kr/board/maple/2300?p=2
2026-08-14 15:35:04,247 [INFO] Page 2: 0개의 신규 게시글 수집 대상 발견
2026-08-14 15:35:04,247 [INFO] 📌 [목록 수집] 3 페이지 접근: https://www.inven.co.kr/board/maple/2300?p=3
2026-08-14 15:35:06,730 [INFO] Page 3: 0개의 신규 게시글 수집 대상 발견
2026-08-14 15:35:06,731 [INFO] 📌 [목록 수집] 4 페이지 접근: https://www.inven.co.kr/board/maple/2300?p=4
2026-08-14 15:35:09,213 [INFO] Page 4: 0개의 신규 게시글 수집 대상 발견
2026-08-14 15:35:09,214 [INFO] 📌 [목록 수집] 5 페이지 접근: https://www.inven.co.kr/board/maple/2300?p=5
2026-08-14 15:35:12,058 [INFO] Page 5: 38개의 신규 게시글 수집 대